# A Neural Probabilistic Language Model

Bengio et al., JMLR 3 (2003) 1137-1155 — [paper (PDF)](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

![Paper title page](images/1-1.png)


In [16]:
import math
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [17]:
class NNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim, use_W=True):
        super(NNLM, self).__init__()
        self.C = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

        # x is the concatenation of context_size embeddings, so it is (n-1) * m wide
        in_features = context_size * embedding_dim

        self.d_Hx = nn.Linear(in_features=in_features, out_features=hidden_dim)

        self.tanh = nn.Tanh()

        self.U = nn.Linear(in_features=hidden_dim, out_features=vocab_size)

        self.use_W = use_W
        if self.use_W:
            # direct connections from x to the output; b is already U's bias
            self.W = nn.Linear(in_features=in_features, out_features=vocab_size, bias=False)


    def forward(self, words):
        # 1 Create Embedding for the words
        embedding = self.C(words)

        # 2. Concatenate the embeddings into vectors like x = C(w_{t-1}) + C(w_{t-2}) + ... + C(w_{t-n+1})
        x = embedding.view(embedding.size(0), -1)

        # 3. Compute the hidden layer representation Hx = tanh(d_Hx * x)
        tanh_d_Hx = self.tanh(self.d_Hx(x))

        # 4. Compute the output layer representation y = U.tanh(Hx + d)
        y = self.U(tanh_d_Hx)

        # 5. If use_W is True, add Wx to the output layer representation y = U.tanh(Hx + d) + Wx
        if self.use_W:
            y = y + self.W(x)

        # raw logits: the softmax of equation (6) is applied by CrossEntropyLoss
        return y

## Data — Penn Treebank

The paper trained on Brown and AP News. PTB is the standard substitute: already tokenized and
lowercased, with rare words replaced by `<unk>`, giving a vocabulary of exactly 10,000 words.

In [18]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
URL = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.{}.txt"


def load_split(name):
    path = DATA_DIR / f"ptb.{name}.txt"
    if not path.exists():
        urllib.request.urlretrieve(URL.format(name), path)
    # one sentence per line; mark the boundary so the model can learn where sentences end
    return path.read_text().replace("\n", " <eos> ").split()


train_words = load_split("train")
valid_words = load_split("valid")
test_words = load_split("test")

print(f"train: {len(train_words)}, first: {train_words[:50]}")

# the vocabulary is fixed by the training set; valid/test already use <unk> for anything else
idx2word = sorted(set(train_words))
print(f"vocabulary {len(idx2word)}: {idx2word[:10]} ... {idx2word[-10:]}")
word2idx = {w: i for i, w in enumerate(idx2word)}
print(f"word2idx: {list(word2idx.items())[:10]} ... {list(word2idx.items())[-10:]}")
UNK = word2idx["<unk>"]
print(f"UNK index: {UNK}")

VOCAB_SIZE = len(idx2word)
print(f"vocabulary {VOCAB_SIZE}, tokens: train {len(train_words)}, valid {len(valid_words)}, test {len(test_words)}")

train: 929589, first: ['aer', 'banknote', 'berlitz', 'calloway', 'centrust', 'cluett', 'fromstein', 'gitano', 'guterman', 'hydro-quebec', 'ipo', 'kia', 'memotec', 'mlx', 'nahb', 'punts', 'rake', 'regatta', 'rubens', 'sim', 'snack-food', 'ssangyong', 'swapo', 'wachter', '<eos>', 'pierre', '<unk>', 'N', 'years', 'old', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', 'N', '<eos>', 'mr.', '<unk>', 'is', 'chairman', 'of', '<unk>', 'n.v.', 'the', 'dutch']
vocabulary 10000: ['#', '$', '&', "'", "'80s", "'d", "'ll", "'m", "'re", "'s"] ... ['zealand', 'zenith', 'zero', 'zero-coupon', 'zeta', 'zip', 'zoete', 'zone', 'zones', 'zurich']
word2idx: [('#', 0), ('$', 1), ('&', 2), ("'", 3), ("'80s", 4), ("'d", 5), ("'ll", 6), ("'m", 7), ("'re", 8), ("'s", 9)] ... [('zealand', 9990), ('zenith', 9991), ('zero', 9992), ('zero-coupon', 9993), ('zeta', 9994), ('zip', 9995), ('zoete', 9996), ('zone', 9997), ('zones', 9998), ('zurich', 9999)]
UNK index: 44
vocabulary 10000, tok

In [20]:
CONTEXT_SIZE = 5  # n - 1, i.e. the paper's n = 6

class PTBDataset(Dataset):
    def __init__(self, words, word2idx, context_size):
        # encode once here instead of on every access, so an epoch is pure tensor slicing
        self.ids = torch.tensor([word2idx.get(word, UNK) for word in words], dtype=torch.long)
        self.context_size = context_size

    def __len__(self):
        return len(self.ids) - self.context_size

    def __getitem__(self, idx):
        context = self.ids[idx:idx + self.context_size]
        target = self.ids[idx + self.context_size]
        return context, target

train_dataset = PTBDataset(train_words, word2idx, CONTEXT_SIZE)
test_dataset = PTBDataset(test_words, word2idx, CONTEXT_SIZE)
val_dataset = PTBDataset(valid_words, word2idx, CONTEXT_SIZE)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


for sentence, label in train_loader:
    print(f"sentence: {sentence}, label: {label}, shape: {sentence.shape}, {label.shape}")
    break

sentence: tensor([[6142, 9018, 6947, 9119,  494],
        [6298,  270,   43, 2294, 8121],
        [6237,    1,   45, 3659, 2897],
        [8524, 2385, 9018,   44, 6185],
        [2420, 3659, 6188, 9958,   48],
        [6142, 9012, 7685,   44,   44],
        [6142, 9012, 7127, 7402, 1291],
        [1283, 9012, 1063,   43, 4470],
        [9012, 7276, 9010, 4748,   43],
        [ 897, 3923,  413, 6224, 9119],
        [ 424, 5057, 3077, 4470,  413],
        [1225, 9838, 2438, 6947, 5798],
        [   9, 9338, 5798, 6778, 6661],
        [  48,   45,   45, 2386, 4470],
        [ 873,   48, 3497, 1417, 9796],
        [3960, 9110,    2, 7766, 1699],
        [9007,    1,   45,   48, 9958],
        [3571, 1352, 4165, 4144,  897],
        [5975, 9977, 9869, 9012, 4661],
        [  48,   45,   45, 1785, 6185],
        [5710, 7824, 4917, 6100, 6188],
        [9811,    9, 5985,   43, 9840],
        [5924, 2791, 9789, 1270, 4081],
        [  45,   48, 8089, 6237,   73],
        [5347,  413,  416, 614

## Training

Hyperparameters follow the paper's MLP setting: `n = 6`, `m = 60` word features, `h = 100`
hidden units, with the direct connections `W` enabled.

In [21]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = NNLM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=60,
    context_size=CONTEXT_SIZE,
    hidden_dim=100,
    use_W=True,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(device, f"{sum(p.numel() for p in model.parameters()):,} parameters")

mps 4,640,100 parameters


In [ ]:
@torch.no_grad()
def evaluate(loader):
    """Mean cross-entropy in nats; perplexity is its exponential."""
    model.eval()
    total, count = 0.0, 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        total += criterion(model(X), Y).item() * Y.size(0)
        count += Y.size(0)
    return total / count


EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, seen = 0.0, 0
    for X, Y in train_loader:
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), Y)
        loss.backward()
        optimizer.step()
        running += loss.item() * Y.size(0)
        seen += Y.size(0)

    val_loss = evaluate(valid_loader)
    print(
        f"epoch {epoch}  train ppl {math.exp(running / seen):7.1f}"
        f"  valid ppl {math.exp(val_loss):7.1f}"
    )

## Evaluation

The paper reports a test perplexity of 252 for the neural model against 312 for the best
smoothed trigram, on Brown. PTB numbers are not directly comparable, but the same gap over
an n-gram baseline should be visible.

In [ ]:
print(f"test perplexity {math.exp(evaluate(test_loader)):.1f}")


@torch.no_grad()
def predict(context, k=5):
    model.eval()
    words = context.split()[-CONTEXT_SIZE:]
    words = ["<eos>"] * (CONTEXT_SIZE - len(words)) + words  # left-pad short contexts
    ids = torch.tensor([[word2idx.get(w, UNK) for w in words]], device=device)
    probs = model(ids).softmax(dim=-1)[0]
    top = probs.topk(k)
    return [(idx2word[i], round(p.item(), 3)) for p, i in zip(top.values, top.indices)]


for context in [
    "the company said it will",
    "the stock market closed",
    "he is the chairman of the",
]:
    print(f"{context} ...")
    for word, prob in predict(context):
        print(f"    {prob:.3f}  {word}")